# SmartBlood — Fixed Multi-Risk Inventory Engine

This notebook is a corrected, dataset-driven version of the uploaded risk-engine notebook.

## What this version changes

1. Loads the historical inventory dataset and feature-engineered forecasting dataset separately.
2. Loads the saved XGBoost model explicitly (the uploaded notebook referenced `model` without loading it).
3. Uses the latest historical `closing_stock` as a **historical stock estimate**, not live hospital inventory.
4. Estimates safety stock from historical daily-demand variability and a configurable replenishment lead time.
5. Estimates emergency feature input using a recent historical average rather than a single latest value.
6. Estimates expiry-wastage exposure from `expired / collected`; it does **not** claim to know individual batch expiry dates.
7. Calculates shortage, excess, expiry, and overall risk.
8. Produces dashboard-ready JSON and includes validation tests.

> Important: Your dataset does not contain batch IDs, collection dates per unit, or individual expiry dates. Exact near-expiry unit counts cannot be calculated. The expiry output is therefore an aggregate historical-wastage proxy.

In [1]:
# =========================================================
# 1. IMPORTS AND CONFIGURATION
# =========================================================

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

# These paths follow the folder structure used in your uploaded notebook.
# If your notebook is stored in a different folder, adjust PROJECT_ROOT.
PROJECT_ROOT = Path("..")

RAW_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "smartblood_dataset_2023_2024_eda_ready.csv"
FEATURE_DATA_PATH = PROJECT_ROOT / "data" / "features" / "blood_demand_features.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "models" / "smartblood_xgboost_model.pkl"

HORIZON_DAYS = 7

# Prototype assumption: replace with the actual hospital replenishment lead time
# when that information becomes available.
LEAD_TIME_DAYS = 2

# Illustrative one-sided 95% service-level factor.
Z_SCORE = 1.645

# Configurable prototype thresholds (not universal medical standards).
SHORTAGE_HIGH_THRESHOLD = 15.0
EXCESS_HIGH_THRESHOLD = 50.0
EXPIRY_HIGH_THRESHOLD = 20.0

# Risk category mapping
RISK_MAPPING = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
REVERSE_RISK_MAPPING = {value: key for key, value in RISK_MAPPING.items()}

# Selected example product. Change to any combination present in your data.
SELECTED_BLOOD_GROUP = "O+"
SELECTED_COMPONENT = "RBC"

# Recent historical records used to estimate the emergency feature input.
EMERGENCY_LOOKBACK_RECORDS = 28

print("Configuration loaded.")
print("Raw data path:", RAW_DATA_PATH)
print("Feature data path:", FEATURE_DATA_PATH)
print("Model path:", MODEL_PATH)

Configuration loaded.
Raw data path: ..\data\processed\smartblood_dataset_2023_2024_eda_ready.csv
Feature data path: ..\data\features\blood_demand_features.csv
Model path: ..\outputs\models\smartblood_xgboost_model.pkl


In [2]:
# =========================================================
# 2. LOAD AND VALIDATE THE HISTORICAL INVENTORY DATASET
# =========================================================

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Historical dataset not found: {RAW_DATA_PATH}\n"
        "Update RAW_DATA_PATH so it points to your cleaned SmartBlood dataset."
    )

inventory_df = pd.read_csv(RAW_DATA_PATH)
inventory_df.columns = inventory_df.columns.str.strip().str.lower()

# Normalize the emergency column name.
if "emergency" not in inventory_df.columns and "emergency_requests" in inventory_df.columns:
    inventory_df = inventory_df.rename(columns={"emergency_requests": "emergency"})

required_inventory_columns = [
    "date", "blood_group", "component", "demand",
    "collected", "issued", "opening_stock",
    "closing_stock", "expired", "emergency"
]

missing = [col for col in required_inventory_columns if col not in inventory_df.columns]
if missing:
    raise ValueError(
        "The historical dataset is missing required columns: "
        + ", ".join(missing)
    )

inventory_df["date"] = pd.to_datetime(inventory_df["date"], errors="coerce")

numeric_columns = [
    "demand", "collected", "issued", "opening_stock",
    "closing_stock", "expired", "emergency"
]
for col in numeric_columns:
    inventory_df[col] = pd.to_numeric(inventory_df[col], errors="coerce")

inventory_df = inventory_df.dropna(subset=["date", "blood_group", "component"])
inventory_df[numeric_columns] = inventory_df[numeric_columns].fillna(0)
inventory_df[numeric_columns] = inventory_df[numeric_columns].clip(lower=0)

inventory_df = inventory_df.sort_values(
    ["blood_group", "component", "date"]
).reset_index(drop=True)

print("Historical inventory dataset loaded.")
print("Shape:", inventory_df.shape)
display(inventory_df.head())

Historical inventory dataset loaded.
Shape: (17544, 29)


,date,blood_group,component,demand,collected,issued,opening_stock,closing_stock,expired,emergency,...,stock_difference,expiry_rate,shortage_risk,stock_coverage_ratio,excess_stock_risk,expiry_risk,emergency_risk,demand_lag_1,demand_lag_7,demand_rolling_7
0,2023-01-01,A+,Plasma,32,31,31,102,102,0,1,...,0,0.000000,0,3.090909,1,0,1,NaN,NaN,NaN
1,2023-01-02,A+,Plasma,35,38,36,102,104,0,9,...,0,0.000000,0,2.888889,1,0,1,32.0,NaN,NaN
2,2023-01-03,A+,Plasma,23,18,23,104,99,0,0,...,0,0.000000,0,4.125000,1,0,0,35.0,NaN,NaN
3,2023-01-04,A+,Plasma,52,48,51,99,95,1,12,...,0,0.006757,0,1.792453,0,1,1,23.0,NaN,NaN
4,2023-01-05,A+,Plasma,32,26,29,95,92,0,2,...,0,0.000000,0,2.787879,1,0,1,52.0,NaN,NaN


In [3]:
# =========================================================
# 3. LOAD FEATURE-ENGINEERED DATA FOR THE XGBOOST MODEL
# =========================================================

if not FEATURE_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Feature-engineered dataset not found: {FEATURE_DATA_PATH}\n"
        "Update FEATURE_DATA_PATH to your blood_demand_features.csv file."
    )

feature_df = pd.read_csv(FEATURE_DATA_PATH)
feature_df.columns = feature_df.columns.str.strip().str.lower()

if "date" not in feature_df.columns:
    raise ValueError("Feature-engineered dataset must contain a 'date' column.")

feature_df["date"] = pd.to_datetime(feature_df["date"], errors="coerce")
feature_df = feature_df.dropna(subset=["date", "blood_group", "component"])
feature_df = feature_df.sort_values(
    ["blood_group", "component", "date"]
).reset_index(drop=True)

print("Feature-engineered dataset loaded.")
print("Shape:", feature_df.shape)
print("Feature columns:", len(feature_df.columns))
display(feature_df.head())

Feature-engineered dataset loaded.
Shape: (16800, 35)
Feature columns: 35


,date,blood_group,component,demand,collected,issued,opening_stock,closing_stock,expired,emergency,...,emergency_risk,demand_lag_1,demand_lag_7,demand_rolling_7,demand_lag_14,demand_lag_30,demand_rolling_14,demand_rolling_30,emergency_requests_lag_1,target_demand
0,2023-01-31,A+,Plasma,29,32,25,145,152,0,2,...,1,23.0,28.0,30.000000,41.0,32.0,28.428571,30.800000,3.0,30.0
1,2023-02-01,A+,Plasma,30,24,29,152,147,0,3,...,1,29.0,30.0,30.142857,26.0,35.0,27.571429,30.700000,2.0,25.0
2,2023-02-02,A+,Plasma,25,18,25,147,140,0,1,...,1,30.0,32.0,30.142857,25.0,23.0,27.857143,30.533333,3.0,26.0
3,2023-02-03,A+,Plasma,26,25,25,140,140,0,1,...,1,25.0,28.0,29.142857,21.0,52.0,27.857143,30.600000,1.0,21.0
4,2023-02-04,A+,Plasma,21,27,21,140,145,1,1,...,1,26.0,32.0,28.857143,26.0,32.0,28.214286,29.733333,1.0,25.0


In [4]:
# =========================================================
# 4. LOAD THE TRAINED XGBOOST MODEL
# =========================================================

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Trained model not found: {MODEL_PATH}\n"
        "Save your trained XGBoost model at this path, or update MODEL_PATH "
        "to the actual .pkl file created by your training notebook."
    )

model = joblib.load(MODEL_PATH)

if hasattr(model, "feature_names_in_"):
    feature_columns = list(model.feature_names_in_)
else:
    raise ValueError(
        "The loaded model does not expose feature_names_in_. "
        "Set feature_columns manually to the exact ordered feature list "
        "used when training this model."
    )

missing_features = [col for col in feature_columns if col not in feature_df.columns]
# Future feature rows are constructed from the model's feature names, so this
# check is informational for features that are generated rather than stored.
print("XGBoost model loaded:", type(model).__name__)
print("Number of model features:", len(feature_columns))
print("Model features not present as dataset columns (may be engineered):", missing_features)

XGBoost model loaded: XGBRegressor
Number of model features: 24
Model features not present as dataset columns (may be engineered): [np.str_('target_day_of_week_num'), np.str_('target_month'), np.str_('target_quarter'), np.str_('target_is_weekend'), np.str_('blood_group_A+'), np.str_('blood_group_A-'), np.str_('blood_group_AB+'), np.str_('blood_group_AB-'), np.str_('blood_group_B+'), np.str_('blood_group_B-'), np.str_('blood_group_O+'), np.str_('blood_group_O-'), np.str_('component_Plasma'), np.str_('component_Platelets'), np.str_('component_RBC')]


In [5]:
# =========================================================
# 5. SELECT AND VALIDATE THE BLOOD PRODUCT
# =========================================================

available_products = (
    inventory_df[["blood_group", "component"]]
    .drop_duplicates()
    .sort_values(["blood_group", "component"])
    .reset_index(drop=True)
)

print("Available blood groups:", sorted(inventory_df["blood_group"].unique()))
print("Available components:", sorted(inventory_df["component"].unique()))
display(available_products)

product_history = feature_df[
    (feature_df["blood_group"] == SELECTED_BLOOD_GROUP)
    & (feature_df["component"] == SELECTED_COMPONENT)
].copy().sort_values("date").reset_index(drop=True)

inventory_history = inventory_df[
    (inventory_df["blood_group"] == SELECTED_BLOOD_GROUP)
    & (inventory_df["component"] == SELECTED_COMPONENT)
].copy().sort_values("date").reset_index(drop=True)

if product_history.empty:
    raise ValueError(
        f"No feature records found for {SELECTED_BLOOD_GROUP} {SELECTED_COMPONENT}."
    )

if inventory_history.empty:
    raise ValueError(
        f"No inventory records found for {SELECTED_BLOOD_GROUP} {SELECTED_COMPONENT}."
    )

if len(product_history) < 31:
    raise ValueError(
        "At least 31 feature-engineered records are required by this notebook's "
        "30-day lag feature logic."
    )

print("Selected product:", SELECTED_BLOOD_GROUP, SELECTED_COMPONENT)
print("Forecast history rows:", len(product_history))
print("Inventory history rows:", len(inventory_history))
print("Latest inventory record:", inventory_history["date"].max())
display(product_history.tail())

Available blood groups: ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']
Available components: ['Plasma', 'Platelets', 'RBC']


,blood_group,component
0,A+,Plasma
1,A+,Platelets
2,A+,RBC
3,A-,Plasma
4,A-,Platelets
5,A-,RBC
6,AB+,Plasma
7,AB+,Platelets
8,AB+,RBC
9,AB-,Plasma


Selected product: O+ RBC
Forecast history rows: 700
Inventory history rows: 731
Latest inventory record: 2024-12-31 00:00:00


,date,blood_group,component,demand,collected,issued,opening_stock,closing_stock,expired,emergency,...,emergency_risk,demand_lag_1,demand_lag_7,demand_rolling_7,demand_lag_14,demand_lag_30,demand_rolling_14,demand_rolling_30,emergency_requests_lag_1,target_demand
695,2024-12-26,O+,RBC,62,82,61,1542,1563,0,2,...,1,61.0,72.0,58.857143,61.0,58.0,59.357143,57.633333,4.0,55.0
696,2024-12-27,O+,RBC,55,54,54,1563,1563,0,2,...,1,62.0,54.0,57.428571,60.0,50.0,59.428571,57.766667,2.0,59.0
697,2024-12-28,O+,RBC,59,70,60,1563,1573,0,2,...,1,55.0,62.0,57.571429,61.0,52.0,59.071429,57.933333,2.0,59.0
698,2024-12-29,O+,RBC,59,47,52,1573,1568,0,1,...,1,59.0,54.0,57.142857,62.0,52.0,58.928571,58.166667,2.0,67.0
699,2024-12-30,O+,RBC,67,71,61,1568,1578,0,8,...,1,59.0,53.0,57.857143,62.0,72.0,58.714286,58.400000,1.0,57.0


In [6]:
# =========================================================
# 6. DATA-DRIVEN INVENTORY ESTIMATORS
# =========================================================

def get_current_inventory(data, blood_group, component):
    """Use the latest recorded closing_stock as a historical stock estimate."""
    history = data[
        (data["blood_group"] == blood_group)
        & (data["component"] == component)
    ].sort_values("date")

    if history.empty:
        raise ValueError(f"No inventory history for {blood_group} {component}.")

    latest = history.iloc[-1]
    return {
        "current_stock": float(latest["closing_stock"]),
        "inventory_date": pd.Timestamp(latest["date"]),
        "inventory_source": "Latest historical closing_stock (not live inventory)"
    }


def estimate_emergency_demand(
    data, blood_group, component, recent_records=EMERGENCY_LOOKBACK_RECORDS
):
    """Estimate daily emergency feature input from recent historical records."""
    history = data[
        (data["blood_group"] == blood_group)
        & (data["component"] == component)
    ].sort_values("date")

    if history.empty:
        raise ValueError(f"No emergency history for {blood_group} {component}.")

    recent = history.tail(recent_records)
    return {
        "average_daily_emergency": float(recent["emergency"].mean()),
        "records_used": int(len(recent)),
        "method": f"Mean of latest {len(recent)} historical records"
    }


def calculate_safety_stock(
    data, blood_group, component,
    lead_time_days=LEAD_TIME_DAYS, z_score=Z_SCORE
):
    """Simplified safety stock: Z * daily demand std * sqrt(lead time)."""
    if lead_time_days <= 0:
        raise ValueError("lead_time_days must be greater than zero.")

    history = data[
        (data["blood_group"] == blood_group)
        & (data["component"] == component)
    ].sort_values("date")

    if history.empty:
        raise ValueError(f"No demand history for {blood_group} {component}.")

    demand_std = float(history["demand"].std(ddof=1))
    if not np.isfinite(demand_std):
        demand_std = 0.0

    safety_stock = z_score * demand_std * np.sqrt(lead_time_days)

    return {
        "daily_demand_std": demand_std,
        "lead_time_days": int(lead_time_days),
        "z_score": float(z_score),
        "safety_stock": float(np.ceil(safety_stock)),
        "method": "Z × daily demand standard deviation × sqrt(lead time)"
    }


def estimate_expiry_exposure(data, blood_group, component):
    """
    Estimate aggregate expiry-wastage exposure using historical expired/collected.
    This is NOT an individual-batch or near-expiry prediction.
    """
    history = data[
        (data["blood_group"] == blood_group)
        & (data["component"] == component)
    ].sort_values("date")

    if history.empty:
        raise ValueError(f"No expiry history for {blood_group} {component}.")

    total_collected = float(history["collected"].sum())
    total_expired = float(history["expired"].sum())

    expiry_rate = (
        total_expired / total_collected
        if total_collected > 0 else 0.0
    )
    expiry_rate = float(np.clip(expiry_rate, 0.0, 1.0))

    stock_info = get_current_inventory(data, blood_group, component)
    estimated_exposure = stock_info["current_stock"] * expiry_rate

    return {
        "historical_expiry_rate": expiry_rate,
        "estimated_expiry_exposure": float(estimated_exposure),
        "method": "Historical expired units / collected units × latest recorded stock",
        "limitation": (
            "Aggregate proxy only; dataset has no batch IDs or individual expiry dates."
        )
    }

print("Inventory estimator functions defined.")

Inventory estimator functions defined.


In [7]:
# =========================================================
# 7. CHECK DATA-DERIVED INVENTORY ESTIMATES
# =========================================================

inventory_estimate = get_current_inventory(
    inventory_df, SELECTED_BLOOD_GROUP, SELECTED_COMPONENT
)

emergency_estimate = estimate_emergency_demand(
    inventory_df, SELECTED_BLOOD_GROUP, SELECTED_COMPONENT
)

safety_stock_estimate = calculate_safety_stock(
    inventory_df, SELECTED_BLOOD_GROUP, SELECTED_COMPONENT
)

expiry_estimate = estimate_expiry_exposure(
    inventory_df, SELECTED_BLOOD_GROUP, SELECTED_COMPONENT
)

print("Latest stock estimate:", inventory_estimate)
print("Recent emergency estimate:", emergency_estimate)
print("Safety stock estimate:", safety_stock_estimate)
print("Expiry exposure estimate:", expiry_estimate)

Latest stock estimate: {'current_stock': 1576.0, 'inventory_date': Timestamp('2024-12-31 00:00:00'), 'inventory_source': 'Latest historical closing_stock (not live inventory)'}
Recent emergency estimate: {'average_daily_emergency': 3.6785714285714284, 'records_used': 28, 'method': 'Mean of latest 28 historical records'}
Safety stock estimate: {'daily_demand_std': 6.441844715734864, 'lead_time_days': 2, 'z_score': 1.645, 'safety_stock': 15.0, 'method': 'Z × daily demand standard deviation × sqrt(lead time)'}
Expiry exposure estimate: {'historical_expiry_rate': 0.007136574946601332, 'estimated_expiry_exposure': 11.247242115843699, 'method': 'Historical expired units / collected units × latest recorded stock', 'limitation': 'Aggregate proxy only; dataset has no batch IDs or individual expiry dates.'}


In [17]:
# =========================================================
# 8. IDENTIFY EMERGENCY FEATURE AND BUILD FUTURE FEATURES
# =========================================================

def find_emergency_feature(feature_names):
    """Find a recognized emergency lag feature used by the trained model."""
    candidates = [
        "emergency_requests_lag_1",
        "emergency_lag_1",
        "emergency_requests",
        "emergency"
    ]
    for name in candidates:
        if name in feature_names:
            return name
    return None


emergency_feature_name = find_emergency_feature(feature_columns)
print("Emergency feature selected:", emergency_feature_name)

def build_future_feature_row(
    demand_history,
    future_date,
    selected_blood_group,
    selected_component,
    emergency_value,
    feature_names
):
    """Build one future row using the same feature names expected by XGBoost."""
    row = {name: 0.0 for name in feature_names}

    # Demand and lag features
    demand_feature_values = {
        "demand": demand_history[-1],
        "demand_lag_1": demand_history[-1],
        "demand_lag_7": demand_history[-7],
        "demand_lag_14": demand_history[-14],
        "demand_lag_30": demand_history[-30],
        "demand_rolling_7": float(np.mean(demand_history[-7:])),
        "demand_rolling_14": float(np.mean(demand_history[-14:])),
        "demand_rolling_30": float(np.mean(demand_history[-30:]))
    }

    for name, value in demand_feature_values.items():
        if name in row:
            row[name] = float(value)

    # Emergency feature: recent historical average, held constant for the
    # recursive forecast because future emergency requests are unknown.
    if emergency_feature_name is not None:
        row[emergency_feature_name] = float(emergency_value)

    # Calendar features used by the uploaded notebook's forecasting logic.
    calendar_values = {
        "target_day_of_week_num": future_date.dayofweek,
        "target_month": future_date.month,
        "target_quarter": future_date.quarter,
        "target_is_weekend": int(future_date.dayofweek >= 5),
        "day_of_week_num": future_date.dayofweek,
        "month": future_date.month,
        "quarter": future_date.quarter,
        "is_weekend": int(future_date.dayofweek >= 5)
    }

    for name, value in calendar_values.items():
        if name in row:
            row[name] = value

    # One-hot encoded product identifiers.
    blood_group_feature = f"blood_group_{selected_blood_group}"
    component_feature = f"component_{selected_component}"

    if blood_group_feature in row:
        row[blood_group_feature] = 1
    if component_feature in row:
        row[component_feature] = 1

    return pd.DataFrame([row], columns=feature_names)

print("Future feature builder defined.")

Emergency feature selected: emergency_requests_lag_1
Future feature builder defined.


In [9]:
# =========================================================
# 9. GENERATE RECURSIVE MULTI-DAY FORECAST
# =========================================================

def generate_future_forecast(
    model,
    product_history,
    selected_blood_group,
    selected_component,
    inventory_history,
    feature_names,
    horizon_days=HORIZON_DAYS
):
    history = product_history.sort_values("date").copy()
    demand_history = history["demand"].astype(float).tolist()
    last_date = pd.Timestamp(history["date"].max())

    emergency_info = estimate_emergency_demand(
        inventory_history,
        selected_blood_group,
        selected_component
    )
    emergency_assumption = emergency_info["average_daily_emergency"]

    forecast_rows = []

    for day in range(1, horizon_days + 1):
        future_date = last_date + pd.Timedelta(days=day)

        X_future = build_future_feature_row(
            demand_history=demand_history,
            future_date=future_date,
            selected_blood_group=selected_blood_group,
            selected_component=selected_component,
            emergency_value=emergency_assumption,
            feature_names=feature_names
        )

        prediction = float(model.predict(X_future)[0])
        prediction = max(0.0, prediction)

        forecast_rows.append({
            "date": future_date,
            "blood_group": selected_blood_group,
            "component": selected_component,
            "predicted_demand": prediction
        })

        # Recursive update for the next forecast day.
        demand_history.append(prediction)

    return pd.DataFrame(forecast_rows), emergency_info


future_forecast, emergency_forecast_assumption = generate_future_forecast(
    model=model,
    product_history=product_history,
    selected_blood_group=SELECTED_BLOOD_GROUP,
    selected_component=SELECTED_COMPONENT,
    inventory_history=inventory_history,
    feature_names=feature_columns,
    horizon_days=HORIZON_DAYS
)

display(future_forecast)

forecast_demand = float(future_forecast["predicted_demand"].clip(lower=0).sum())

print(f"Total predicted demand for {HORIZON_DAYS} days: {forecast_demand:.2f} units")
print(
    "Emergency feature assumption:",
    emergency_forecast_assumption["average_daily_emergency"],
    "average requests per historical record"
)

,date,blood_group,component,predicted_demand
0,2024-12-31,O+,RBC,54.688526
1,2025-01-01,O+,RBC,57.301079
2,2025-01-02,O+,RBC,55.525707
3,2025-01-03,O+,RBC,57.254101
4,2025-01-04,O+,RBC,60.444946
5,2025-01-05,O+,RBC,60.099873
6,2025-01-06,O+,RBC,54.382416


Total predicted demand for 7 days: 399.70 units
Emergency feature assumption: 3.6785714285714284 average requests per historical record


In [10]:
# =========================================================
# 10. RISK CLASSIFICATION AND OVERALL RISK
# =========================================================

def classify_shortage_risk(shortage_gap, high_threshold=SHORTAGE_HIGH_THRESHOLD):
    if shortage_gap <= 0:
        return "LOW"
    if shortage_gap >= high_threshold:
        return "HIGH"
    return "MEDIUM"


def classify_excess_risk(excess_units, high_threshold=EXCESS_HIGH_THRESHOLD):
    if excess_units <= 0:
        return "LOW"
    if excess_units >= high_threshold:
        return "HIGH"
    return "MEDIUM"


def classify_expiry_risk(estimated_expiry_exposure, high_threshold=EXPIRY_HIGH_THRESHOLD):
    if estimated_expiry_exposure <= 0:
        return "LOW"
    if estimated_expiry_exposure >= high_threshold:
        return "HIGH"
    return "MEDIUM"


def calculate_overall_risk(shortage_risk, excess_risk, expiry_risk):
    risks = [shortage_risk, excess_risk, expiry_risk]

    for risk in risks:
        if risk not in RISK_MAPPING:
            raise ValueError(f"Invalid risk category: {risk}")

    overall_score = max(RISK_MAPPING[risk] for risk in risks)
    overall_risk = REVERSE_RISK_MAPPING[overall_score]
    return overall_score, overall_risk

print("Risk classification functions defined.")

Risk classification functions defined.


In [11]:
# =========================================================
# 11. COMPLETE MULTI-RISK INVENTORY ASSESSMENT
# =========================================================

def assess_blood_inventory(
    inventory_data,
    blood_group,
    component,
    forecast_demand_7_days,
    lead_time_days=LEAD_TIME_DAYS
):
    if forecast_demand_7_days < 0:
        raise ValueError("Forecast demand cannot be negative.")

    stock_info = get_current_inventory(inventory_data, blood_group, component)
    safety_info = calculate_safety_stock(
        inventory_data, blood_group, component,
        lead_time_days=lead_time_days
    )
    emergency_info = estimate_emergency_demand(
        inventory_data, blood_group, component
    )
    expiry_info = estimate_expiry_exposure(
        inventory_data, blood_group, component
    )

    current_stock = stock_info["current_stock"]
    safety_stock = safety_info["safety_stock"]
    target_stock = float(forecast_demand_7_days + safety_stock)

    # Risk 1: shortage
    shortage_gap = max(0.0, target_stock - current_stock)
    projected_stock_after_forecast = current_stock - forecast_demand_7_days
    shortage_risk = classify_shortage_risk(shortage_gap)

    # Risk 2: excess
    excess_units = max(0.0, current_stock - target_stock)
    excess_risk = classify_excess_risk(excess_units)

    # Risk 3: aggregate expiry-wastage exposure proxy
    estimated_expiry_exposure = expiry_info["estimated_expiry_exposure"]
    expiry_risk = classify_expiry_risk(estimated_expiry_exposure)

    overall_score, overall_risk = calculate_overall_risk(
        shortage_risk, excess_risk, expiry_risk
    )

    return {
        "blood_group": blood_group,
        "component": component,
        "inventory_date": stock_info["inventory_date"].strftime("%Y-%m-%d"),
        "inventory_source": stock_info["inventory_source"],
        "current_stock": round(float(current_stock), 2),
        "forecast_demand_7_days": round(float(forecast_demand_7_days), 2),
        "projected_stock_after_forecast": round(float(projected_stock_after_forecast), 2),
        "lead_time_days_assumption": int(lead_time_days),
        "safety_stock": round(float(safety_stock), 2),
        "target_stock_for_horizon": round(float(target_stock), 2),
        "shortage_gap": round(float(shortage_gap), 2),
        "shortage_risk": shortage_risk,
        "excess_units": round(float(excess_units), 2),
        "excess_risk": excess_risk,
        "average_daily_emergency_requests_recent_history": round(
            float(emergency_info["average_daily_emergency"]), 2
        ),
        "emergency_estimation_method": emergency_info["method"],
        "historical_expiry_rate": round(
            float(expiry_info["historical_expiry_rate"]), 6
        ),
        "estimated_expiry_wastage_exposure": round(
            float(estimated_expiry_exposure), 2
        ),
        "expiry_risk": expiry_risk,
        "expiry_method": expiry_info["method"],
        "expiry_limitation": expiry_info["limitation"],
        "overall_score": int(overall_score),
        "overall_risk": overall_risk,
        "data_mode": "Historical SmartBlood dataset prototype"
    }

risk_result = assess_blood_inventory(
    inventory_data=inventory_df,
    blood_group=SELECTED_BLOOD_GROUP,
    component=SELECTED_COMPONENT,
    forecast_demand_7_days=forecast_demand,
    lead_time_days=LEAD_TIME_DAYS
)

print("SmartBlood Multi-Risk Assessment")
print("=" * 55)
for key, value in risk_result.items():
    print(f"{key}: {value}")

SmartBlood Multi-Risk Assessment
blood_group: O+
component: RBC
inventory_date: 2024-12-31
inventory_source: Latest historical closing_stock (not live inventory)
current_stock: 1576.0
forecast_demand_7_days: 399.7
projected_stock_after_forecast: 1176.3
lead_time_days_assumption: 2
safety_stock: 15.0
target_stock_for_horizon: 414.7
shortage_gap: 0.0
shortage_risk: LOW
excess_units: 1161.3
excess_risk: HIGH
average_daily_emergency_requests_recent_history: 3.68
emergency_estimation_method: Mean of latest 28 historical records
historical_expiry_rate: 0.007137
estimated_expiry_wastage_exposure: 11.25
expiry_risk: MEDIUM
expiry_method: Historical expired units / collected units × latest recorded stock
expiry_limitation: Aggregate proxy only; dataset has no batch IDs or individual expiry dates.
overall_score: 2
overall_risk: HIGH
data_mode: Historical SmartBlood dataset prototype


In [12]:
# =========================================================
# 12. GENERATE EXPLANATIONS
# =========================================================

def generate_risk_explanations(result):
    explanations = {}

    if result["shortage_risk"] == "HIGH":
        explanations["shortage"] = (
            "High shortage risk: the forecast plus safety-stock target "
            "exceeds the latest recorded stock by a substantial amount."
        )
    elif result["shortage_risk"] == "MEDIUM":
        explanations["shortage"] = (
            "Medium shortage risk: the latest recorded stock is below "
            "the forecast plus safety-stock target."
        )
    else:
        explanations["shortage"] = (
            "Low shortage risk: the latest recorded stock meets the "
            "forecast plus safety-stock target under the prototype rule."
        )

    if result["excess_risk"] == "HIGH":
        explanations["excess"] = (
            "High excess-stock risk: recorded stock is substantially "
            "above the forecast-horizon target."
        )
    elif result["excess_risk"] == "MEDIUM":
        explanations["excess"] = (
            "Medium excess-stock risk: recorded stock is above the "
            "forecast-horizon target."
        )
    else:
        explanations["excess"] = (
            "Low excess-stock risk: no excess was detected by the "
            "forecast-horizon comparison."
        )

    if result["expiry_risk"] == "HIGH":
        explanations["expiry"] = (
            "High expiry-wastage exposure proxy based on the historical "
            "expired-to-collected ratio. This is not a batch-level expiry prediction."
        )
    elif result["expiry_risk"] == "MEDIUM":
        explanations["expiry"] = (
            "Medium expiry-wastage exposure proxy based on historical "
            "expiry patterns. Individual expiry dates are unavailable."
        )
    else:
        explanations["expiry"] = (
            "Low historical expiry-wastage exposure under the current proxy; "
            "this does not confirm that no individual units are near expiry."
        )

    explanations["overall"] = (
        "Overall risk is the maximum severity among shortage, excess, "
        "and expiry risk categories. The score is an ordinal severity "
        "level (0=LOW, 1=MEDIUM, 2=HIGH), not a probability."
    )

    return explanations


risk_explanations = generate_risk_explanations(risk_result)
display(risk_explanations)

{'shortage': 'Low shortage risk: the latest recorded stock meets the forecast plus safety-stock target under the prototype rule.',
 'excess': 'High excess-stock risk: recorded stock is substantially above the forecast-horizon target.',
 'expiry': 'Medium expiry-wastage exposure proxy based on historical expiry patterns. Individual expiry dates are unavailable.',
 'overall': 'Overall risk is the maximum severity among shortage, excess, and expiry risk categories. The score is an ordinal severity level (0=LOW, 1=MEDIUM, 2=HIGH), not a probability.'}

In [13]:
# =========================================================
# 13. DASHBOARD-READY JSON OUTPUT
# =========================================================

dashboard_output = {
    **risk_result,
    "planning_horizon_days": int(HORIZON_DAYS),
    "forecast_daily_values": [
        {
            "date": row["date"].strftime("%Y-%m-%d"),
            "predicted_demand": round(float(row["predicted_demand"]), 2)
        }
        for _, row in future_forecast.iterrows()
    ],
    "explanations": risk_explanations
}

dashboard_json = json.dumps(dashboard_output, indent=4)
print(dashboard_json)

# Save a local JSON response for backend/frontend integration.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "risk_engine"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON_PATH = OUTPUT_DIR / "smartblood_risk_result.json"

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as file:
    file.write(dashboard_json)

print("\nDashboard JSON saved to:", OUTPUT_JSON_PATH)

{
    "blood_group": "O+",
    "component": "RBC",
    "inventory_date": "2024-12-31",
    "inventory_source": "Latest historical closing_stock (not live inventory)",
    "current_stock": 1576.0,
    "forecast_demand_7_days": 399.7,
    "projected_stock_after_forecast": 1176.3,
    "lead_time_days_assumption": 2,
    "safety_stock": 15.0,
    "target_stock_for_horizon": 414.7,
    "shortage_gap": 0.0,
    "shortage_risk": "LOW",
    "excess_units": 1161.3,
    "excess_risk": "HIGH",
    "average_daily_emergency_requests_recent_history": 3.68,
    "emergency_estimation_method": "Mean of latest 28 historical records",
    "historical_expiry_rate": 0.007137,
    "estimated_expiry_wastage_exposure": 11.25,
    "expiry_risk": "MEDIUM",
    "expiry_method": "Historical expired units / collected units \u00d7 latest recorded stock",
    "expiry_limitation": "Aggregate proxy only; dataset has no batch IDs or individual expiry dates.",
    "overall_score": 2,
    "overall_risk": "HIGH",
    "dat

In [14]:
# =========================================================
# 14. TEST OVERALL RISK LOGIC — ALL 27 COMBINATIONS
# =========================================================

from itertools import product

risk_levels = ["LOW", "MEDIUM", "HIGH"]
test_count = 0

for shortage, excess, expiry in product(risk_levels, repeat=3):
    score, category = calculate_overall_risk(shortage, excess, expiry)
    expected_score = max(
        RISK_MAPPING[shortage],
        RISK_MAPPING[excess],
        RISK_MAPPING[expiry]
    )
    assert score == expected_score
    assert category == REVERSE_RISK_MAPPING[expected_score]
    test_count += 1

print("Total combinations tested:", test_count)
print("All 27 overall-risk combinations passed.")

Total combinations tested: 27
All 27 overall-risk combinations passed.


In [15]:
# =========================================================
# 15. TEST INVALID RISK INPUT
# =========================================================

try:
    calculate_overall_risk(
        shortage_risk="LOW",
        excess_risk="CRITICAL",
        expiry_risk="HIGH"
    )
except ValueError as error:
    print("Invalid-risk validation passed:")
    print(error)
else:
    raise AssertionError("Invalid risk category was not rejected.")

Invalid-risk validation passed:
Invalid risk category: CRITICAL


In [16]:
# =========================================================
# 16. FINAL DATA QUALITY / ASSUMPTION CHECK
# =========================================================

print("FINAL SMARTBLOOD PROTOTYPE CHECK")
print("-" * 55)
print("Selected product:", SELECTED_BLOOD_GROUP, SELECTED_COMPONENT)
print("Forecast horizon:", HORIZON_DAYS, "days")
print("Inventory date:", risk_result["inventory_date"])
print("Inventory source:", risk_result["inventory_source"])
print("Emergency estimate:", risk_result["average_daily_emergency_requests_recent_history"])
print("Safety-stock lead-time assumption:", LEAD_TIME_DAYS, "days")
print("Expiry method:", risk_result["expiry_method"])
print("Expiry limitation:", risk_result["expiry_limitation"])
print("Overall risk:", risk_result["overall_risk"], "(score:", risk_result["overall_score"], ")")

assert risk_result["overall_risk"] in RISK_MAPPING
assert risk_result["overall_score"] in [0, 1, 2]
assert risk_result["current_stock"] >= 0
assert risk_result["forecast_demand_7_days"] >= 0
assert risk_result["estimated_expiry_wastage_exposure"] >= 0

print("\nCore output validation passed.")

FINAL SMARTBLOOD PROTOTYPE CHECK
-------------------------------------------------------
Selected product: O+ RBC
Forecast horizon: 7 days
Inventory date: 2024-12-31
Inventory source: Latest historical closing_stock (not live inventory)
Emergency estimate: 3.68
Safety-stock lead-time assumption: 2 days
Expiry method: Historical expired units / collected units × latest recorded stock
Expiry limitation: Aggregate proxy only; dataset has no batch IDs or individual expiry dates.
Overall risk: HIGH (score: 2 )

Core output validation passed.


## Interpretation and limitations for your report

- `current_stock` is the latest `closing_stock` in the historical dataset. It is not live hospital stock.
- `safety_stock` uses a configurable assumed replenishment lead time. Replace it with a hospital-supplied lead time when available.
- The emergency feature uses the average of recent historical emergency records because future emergency requests are unknown. This is a prototype assumption, not a separate emergency forecast.
- `estimated_expiry_wastage_exposure` is derived from the historical `expired / collected` ratio multiplied by the latest recorded stock. It is not an actual count of near-expiry units.
- Exact batch-level expiry analysis requires batch IDs, unit quantities, collection dates, and expiry dates from a hospital inventory system.
- The overall score is an ordinal maximum-severity score: 0 = LOW, 1 = MEDIUM, 2 = HIGH. It is not a probability.

## Before running

Make sure these three files exist at the configured paths:
1. `data/processed/smartblood_dataset_2023_2024_eda_ready.csv`
2. `data/features/blood_demand_features.csv`
3. `outputs/models/smartblood_xgboost_model.pkl`

Run all cells from top to bottom. If your saved model has a different filename or folder, update `MODEL_PATH` in Cell 1.